In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Einen Job überwachen oder abbrechen

Sieh dir eine Liste deiner Workloads auf der [Workloads-Seite](https://quantum.cloud.ibm.com/workloads) an.

## Job-Status anzeigen
Gehe zu deiner [Workloads-Tabelle](https://quantum.cloud.ibm.com/workloads) und prüfe in der Spalte „Status", ob ein Job abgeschlossen wurde oder fehlgeschlagen ist.

## Verbleibendes Kontingent anzeigen
Gehe zu deiner [Instanzen-Tabelle](https://quantum.cloud.ibm.com/instances) und wähle den Tab aus, der dem Plan entspricht, für den du das verbleibende Kontingent anzeigen möchtest. Die insgesamt genutzte Zeit und die verbleibende Zeit in deinem Plan werden angezeigt.

## Metriken zur Anzahl der eingereichten Jobs und Workloads anzeigen
Gehe zur [Analytics-Seite](https://quantum.cloud.ibm.com/analytics), um die Gesamtzahl der eingereichten Jobs sowie die Anzahl der Batch-Workloads und Session-Workloads zu sehen. Beachte, dass du die Analytics-Seite nur für Konten anzeigen kannst, die du besitzt oder verwaltest.

## Einen Job überwachen
Verwende die Job-Instanz, um den Job-Status zu prüfen oder die Ergebnisse abzurufen, indem du den passenden Befehl aufrufst:

|                               |                                                                                                                                                                                                                                     |
| ----------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| job.result()                  | Sieh dir die Job-Ergebnisse unmittelbar nach Abschluss des Jobs an. Job-Ergebnisse sind verfügbar, sobald der Job abgeschlossen ist. Daher ist job.result() ein blockierender Aufruf, bis der Job abgeschlossen ist.                |
| job.job\_id()                 | Gibt die ID zurück, die diesen Job eindeutig identifiziert. Um Job-Ergebnisse zu einem späteren Zeitpunkt abzurufen, wird die Job-ID benötigt. Es wird daher empfohlen, die IDs von Jobs zu speichern, die du später abrufen möchtest. |
| job.status()                  | Prüfe den Job-Status.                                                                                                                                                                                                               |
| job = service.job(\<job\_id>) | Rufe einen zuvor eingereichten Job ab. Dieser Aufruf erfordert die Job-ID.                                                                                                                                                          |

<span id="retrieve-later"></span>
## Job-Ergebnisse zu einem späteren Zeitpunkt abrufen
Rufe `service.job(\<job\_id>)` auf, um einen zuvor eingereichten Job abzurufen. Wenn du die Job-ID nicht hast oder mehrere Jobs auf einmal abrufen möchtest – einschließlich Jobs von außer Betrieb genommenen QPUs (Quantenprozessoren) –, rufe stattdessen `service.jobs()` mit optionalen Filtern auf. Siehe [QiskitRuntimeService.jobs](../api/qiskit-ibm-runtime/qiskit-runtime-service#jobs).

> **Note:** `service.jobs()` gibt auch Jobs zurück, die mit dem veralteten Paket `qiskit-ibm-provider` ausgeführt wurden. Jobs, die mit dem noch älteren (ebenfalls veralteten) Paket `qiskit-ibmq-provider` eingereicht wurden, sind nicht mehr verfügbar.

### Beispiel
Dieses Beispiel gibt die 10 neuesten Runtime-Jobs zurück, die auf `my_backend` ausgeführt wurden:

In [ ]:
# This cell is hidden from users
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import numpy as np


my_backend = "ibm_torino"
service = QiskitRuntimeService()
# backend = service.backend(my_backend)
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

params = np.random.uniform(size=(2, 3)).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = SamplerV2(mode=backend)
job = sampler.run([sampler_pub], shots=4)
print(job.job_id())

d305ck0ocacs73ajagvg


In [2]:
result = job.result()


spans = job.result().metadata["execution"]["execution_spans"]
print(spans)

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])


In [3]:
params = np.random.uniform(size=(2, 3))
params

array([[0.2260416 , 0.8747859 , 0.44361995],
       [0.94700856, 0.96826017, 0.98426562]])

In [4]:
mask = spans[0].mask(0)
mask

array([[[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]]])

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the account first.
service = QiskitRuntimeService()
# Use `limit` to retrieve a specific number of jobs. The default `limit` is 10.
service.jobs(backend_name=my_backend)

## Cancel a job

You can cancel a job from the IBM Quantum Platform dashboard either on the Workloads page or the details page for a specific workload. On the Workloads page, click the overflow menu at the end of the row for that workload, and select Cancel. If you are on the details page for a specific workload, use the Actions dropdown at the top of the page, and select Cancel.

In Qiskit, use `job.cancel()` to cancel a job.

## Next steps

<Admonition type="tip" title="Recommendations">
    - Try the [Grover's algorithm](/docs/tutorials/grovers-algorithm) tutorial.
    - Learn more about [Sampler execution spans](/docs/guides/sampler-input-output#execution-spans)
</Admonition>